## Q1. How many lesson pages

First, we pull the lesson pages straight from the course repository. We use commit `8c1834d` so everyone works with the exact same data.

How many lesson pages are in the dataset?

* 24
* 72
* 240
* 720

In [1]:
from gitsource import GithubRepositoryDataReader


reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(len(documents))

72


## Q2. Indexing and searching

Index the documents with `minsearch` using `content` as a text field and `filename` as a keyword field.

Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

* `01-agentic-rag/lessons/03-rag.md`
* `01-agentic-rag/lessons/14-agentic-loop.md`
* `04-evaluation/lessons/13-llm-as-judge.md`
* `06-best-practices/lessons/02-hybrid-search.md`

In [2]:
from minsearch import Index


index = Index(
    text_fields=["content"],
    keyword_fields=["filename"])

index.fit(documents)

search_results = index.search(
    query="How does the agentic loop keep calling the model until it stops?"
)

print(search_results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

Now we build a RAG assistant on top of this data.

Build a RAG over the index from Q2 and answer this query:

> How does the agentic loop keep calling the model until it stops?

Use `OPENAI_MODEL_NAME`, which defaults to `gpt-5.4-mini`.

How many input tokens did we send to the model for this request?

* 700
* 7000
* 70000
* 700000

In [3]:
import os

from homework_rag_helper import RAGBase

from openai import OpenAI


openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

agent = RAGBase(
    index=index,
    llm_client=openai_client,
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.4-mini")
)

# I decided to keep the usage shown as part of the rag function call
# instead of complicating the return type.
response = agent.rag("How does the agentic loop keep calling the model until it stops?")

print(response)

ResponseUsage(input_tokens=7136, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=103, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7239)
It keeps calling the model inside a `while True` loop.

Each iteration:
- sends the full `messages` history to the model,
- checks the response for any `function_call`s,
- runs the tools and appends their outputs,
- then calls the model again.

It stops when the model returns no function calls:

```python
if has_function_calls == False:
    break
```

So the loop continues until the model gives a final answer without asking for any more tools.


## Q4. Chunking

Long lesson pages make retrieval less precise, so we split them into overlapping chunks.

Use `chunk_documents` with `size=2000` and `step=1000`.

How many chunks do you get?

* 70
* 295
* 1100
* 4500

In [4]:
from gitsource import chunk_documents


chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

## Q5. RAG with chunking

Chunking makes each request smaller because we send a smaller context to the LLM.

Index the chunks from Q4 with `content` as a text field and `filename` as a keyword field, then answer the same query again and compare the input tokens with Q3.

How many fewer input tokens does the chunked version send?

* about the same
* 3× fewer
* 10× fewer
* 30× fewer

In [5]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"])

index.fit(chunks)


openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

agent = RAGBase(
    index=index,
    llm_client=openai_client,
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.4-mini")
)

response = agent.rag("How does the agentic loop keep calling the model until it stops?")

ResponseUsage(input_tokens=2319, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=100, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2419)


## Q6. Turning it into an agent

So far search runs once with the exact query. Now we make it agentic by giving the LLM a `search` tool and letting it decide when and what to search.

Use these instructions for the agent:

> You're a course teaching assistant.
> Answer the student's question using the search tool.
> Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

How many times did the agent call `search`?

* 0
* 4
* 10
* 20

In [6]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback


instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()


def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    boost_dict = {'filename': 2.0, 'content': 1.0}
    # filter_dict = {'course': self.course}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        # filter_dict=filter_dict
    )


tools = Tools()
tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.4-mini"))
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

print(result.cost)

-> Response received


-> Response received


CostInfo(input_cost=Decimal('0.00601425'), output_cost=Decimal('0.0022995'), total_cost=Decimal('0.00831375'))
